Import Statements

In [1]:
import pandas as pd
import numpy as np
import os   

Load Data

In [2]:
mar_fut_df = pd.read_csv('march_fut.csv')
feb_fut_df = pd.read_csv('feb_fut.csv')

feb_fut_df['Date']   = pd.to_datetime(feb_fut_df['Date'],   format='%d-%m-%Y')
feb_fut_df['Expiry'] = pd.to_datetime(feb_fut_df['Expiry'], format='%d-%m-%Y')

mar_fut_df['Date']   = pd.to_datetime(mar_fut_df['Date'],   format='%d-%m-%Y')
mar_fut_df['Expiry'] = pd.to_datetime(mar_fut_df['Expiry'], format='%d-%m-%Y')  

feb_fut_df = feb_fut_df.sort_values('Date').reset_index(drop=True)
mar_fut_df = mar_fut_df.sort_values('Date').reset_index(drop=True)

feb_fut_df = feb_fut_df[['Date', 'Expiry', 'Underlying Value', 'Settle Price']]
feb_fut_df = feb_fut_df.rename(columns={
    'Underlying Value' : 'Spot_Price',
    'Settle Price'     : 'Feb_Actual'
})

mar_fut_df = mar_fut_df[['Date', 'Expiry', 'Underlying Value', 'Settle Price']]
mar_fut_df = mar_fut_df.rename(columns={
    'Underlying Value' : 'Spot_Price',
    'Settle Price'     : 'Mar_Actual'
})

print("Feb Futures Data:")
print(feb_fut_df.head())
print("\nMar Futures Data:")
print(mar_fut_df.head())

df = feb_fut_df[['Date', 'Spot_Price', 'Feb_Actual']].merge(
     mar_fut_df[['Date', 'Mar_Actual']],
     on='Date', how='outer'
).sort_values('Date').reset_index(drop=True)

print("\nMerged Data:")
print(df.head())


Feb Futures Data:
        Date     Expiry  Spot_Price  Feb_Actual
0 2026-01-01 2026-02-24      991.15     1001.00
1 2026-01-02 2026-02-24     1001.60     1010.75
2 2026-01-05 2026-02-24      977.50      985.75
3 2026-01-06 2026-02-24      962.20      972.30
4 2026-01-07 2026-02-24      949.05      958.00

Mar Futures Data:
        Date     Expiry  Spot_Price  Mar_Actual
0 2026-01-01 2026-03-30      991.15     1007.40
1 2026-01-02 2026-03-30     1001.60     1017.05
2 2026-01-05 2026-03-30      977.50      992.05
3 2026-01-06 2026-03-30      962.20      978.90
4 2026-01-07 2026-03-30      949.05      963.80

Merged Data:
        Date  Spot_Price  Feb_Actual  Mar_Actual
0 2026-01-01      991.15     1001.00     1007.40
1 2026-01-02     1001.60     1010.75     1017.05
2 2026-01-05      977.50      985.75      992.05
3 2026-01-06      962.20      972.30      978.90
4 2026-01-07      949.05      958.00      963.80


FEB/MARCH 2026 THEORETICAL FUTURES PRICE

In [ ]:
feb_expiry = pd.Timestamp('2026-02-26')
mar_expiry = pd.Timestamp('2026-03-30')

def get_rate_91(pricing_date):
    """
    Returns the 91-Day rate based explicitly on the date ranges 
    provided in the January 2026 comments.
    """
    # Ensure the date is a pandas Timestamp to safely access .year, .month, and .day
    t = pd.to_datetime(pricing_date)
    
    # Check if the date falls in January 2026
    if t.year == 2026 and t.month == 1:
        day = t.day
        if 1 <= day <= 6:
            return 0.052237   # 1-6  Jan
        elif 7 <= day <= 13:
            return 0.052761   # 7-13 Jan
        elif 14 <= day <= 20:
            return 0.053094   # 14-20 Jan
        elif 21 <= day <= 27:
            return 0.054517   # 21-27 Jan
        elif 28 <= day <= 31:
            return 0.054623   # 28-31 Jan
            
    # Fallback for any dates outside of the strictly defined January ranges
    return np.nan 

theo_feb      = []
theo_mar      = []
diff_feb      = []
diff_mar      = []
days_feb_list = []
days_mar_list = []
rate_91_list  = []

# Iterate through the dataframe
for _, row in df.iterrows():
    t = pd.to_datetime(row['Date'])
    S = row['Spot_Price']
    r_91 = get_rate_91(t)

    days_feb = (feb_expiry - t).days
    days_mar = (mar_expiry - t).days

    # Only calculate theoretical futures if a valid interest rate is found
    if pd.notna(r_91):
        F_feb = S * np.exp(r_91 * days_feb / 365)
        F_mar = S * np.exp(r_91 * days_mar / 365)
    else:
        F_feb = np.nan
        F_mar = np.nan

    # Calculate differences, checking for valid (non-NaN) Actuals and Theoreticals
    b_feb = row['Feb_Actual'] - F_feb if pd.notna(row['Feb_Actual']) and pd.notna(F_feb) else np.nan
    b_mar = row['Mar_Actual'] - F_mar if pd.notna(row['Mar_Actual']) and pd.notna(F_mar) else np.nan

    # Append rounded results (or NaNs) to lists
    theo_feb.append(round(F_feb,  2) if pd.notna(F_feb) else np.nan)
    theo_mar.append(round(F_mar,  2) if pd.notna(F_mar) else np.nan)
    diff_feb.append(round(b_feb,  2) if pd.notna(b_feb) else np.nan)
    diff_mar.append(round(b_mar,  2) if pd.notna(b_mar) else np.nan)
    days_feb_list.append(days_feb)
    days_mar_list.append(days_mar)
    rate_91_list.append(r_91)

# Compile final DataFrame
df_2026 = df[['Date', 'Spot_Price', 'Feb_Actual', 'Mar_Actual']].copy()
df_2026['r_91']      = rate_91_list
df_2026['days_Feb']  = days_feb_list
df_2026['Theo_Feb']  = theo_feb
df_2026['Diff_Feb']  = diff_feb
df_2026['days_Mar']  = days_mar_list
df_2026['Theo_Mar']  = theo_mar
df_2026['Diff_Mar']  = diff_mar

# Reorder columns
df_2026 = df_2026[[
    'Date',
    'Spot_Price',
    'r_91',
    'days_Feb',
    'Theo_Feb',
    'Feb_Actual',
    'Diff_Feb',
    'days_Mar',
    'Theo_Mar',
    'Mar_Actual',
    'Diff_Mar'
]]

print("\n2026 Futures Output:")
print(df_2026.to_string())


2026 Futures Output:
         Date  Spot_Price      r_91  days_Feb  Theo_Feb  Feb_Actual  Diff_Feb  days_Mar  Theo_Mar  Mar_Actual  Diff_Mar
0  2026-01-01      991.15  0.052237        56    999.13     1001.00      1.87        88   1003.71     1007.40      3.69
1  2026-01-02     1001.60  0.052237        55   1009.52     1010.75      1.23        87   1014.15     1017.05      2.90
2  2026-01-05      977.50  0.052237        52    984.80      985.75      0.95        84    989.32      992.05      2.73
3  2026-01-06      962.20  0.052237        51    969.25      972.30      3.05        83    973.70      978.90      5.20
4  2026-01-07      949.05  0.052761        50    955.93      958.00      2.07        82    960.37      963.80      3.43
5  2026-01-08      946.70  0.052761        49    953.43      956.05      2.62        81    957.85      962.45      4.60
6  2026-01-09      939.00  0.052761        48    945.54      948.10      2.56        80    949.92      954.50      4.58
7  2026-01-12     

FEB 2027 THEORETICAL FUTURES PRICE

In [6]:
feb27_expiry = pd.Timestamp('2027-02-04')

# 364-Day Rc — for Feb 2027 futures cost of carry
rbi_rates_364 = {
    pd.Timestamp('2026-01-01') : 0.053911,   # 1-6  Jan
    pd.Timestamp('2026-01-07') : 0.054309,   # 7-13 Jan
    pd.Timestamp('2026-01-14') : 0.054755,   # 14-20 Jan
    pd.Timestamp('2026-01-21') : 0.055628,   # 21-27 Jan
    pd.Timestamp('2026-01-28') : 0.056319,   # 28-31 Jan
}

# 182-Day Rc — for discounting dividends
rbi_rates_182 = {
    pd.Timestamp('2026-01-01') : 0.054021,   # 1-6  Jan
    pd.Timestamp('2026-01-07') : 0.054620,   # 7-13 Jan
    pd.Timestamp('2026-01-14') : 0.055195,   # 14-20 Jan
    pd.Timestamp('2026-01-21') : 0.055843,   # 21-27 Jan
    pd.Timestamp('2026-01-28') : 0.056004,   # 28-31 Jan
}

def get_rate_364(pricing_date):
    available = {k: v for k, v in rbi_rates_364.items() if k <= pricing_date}
    if not available:
        return list(rbi_rates_364.values())[0]
    return available[max(available.keys())]

def get_rate_182(pricing_date):
    available = {k: v for k, v in rbi_rates_182.items() if k <= pricing_date}
    if not available:
        return list(rbi_rates_182.values())[0]
    return available[max(available.keys())]

# Dividend inputs
D1        = 22
D2        = 5
ex_date_1 = pd.Timestamp('2026-06-27')
ex_date_2 = pd.Timestamp('2026-07-25')

theo_feb27      = []
days_feb27_list = []
rate_364_list   = []
rate_182_list   = []
pv_d1_list      = []
pv_d2_list      = []
pv_d_list       = []

for _, row in df.iterrows():

    t = row['Date']
    S = row['Spot_Price']

    r_364 = get_rate_364(t)
    r_182 = get_rate_182(t)

    days_feb27 = (feb27_expiry - t).days
    days_d1    = (ex_date_1    - t).days
    days_d2    = (ex_date_2    - t).days

    PV_D1 = D1 * np.exp(-r_182 * days_d1 / 365)
    PV_D2 = D2 * np.exp(-r_182 * days_d2 / 365)
    PV_D  = PV_D1 + PV_D2

    F_feb27 = (S - PV_D) * np.exp(r_364 * days_feb27 / 365)

    theo_feb27.append(round(F_feb27, 2))
    days_feb27_list.append(days_feb27)
    rate_364_list.append(r_364)
    rate_182_list.append(r_182)
    pv_d1_list.append(round(PV_D1,   2))
    pv_d2_list.append(round(PV_D2,   2))
    pv_d_list.append(round(PV_D,     2))

df_2027 = df[['Date', 'Spot_Price']].copy()
df_2027['r_364']      = rate_364_list
df_2027['r_182']      = rate_182_list
df_2027['days_Feb27'] = days_feb27_list
df_2027['PV_D1']      = pv_d1_list
df_2027['PV_D2']      = pv_d2_list
df_2027['PV_D']       = pv_d_list
df_2027['Theo_Feb27'] = theo_feb27

df_2027 = df_2027[[
    'Date',
    'Spot_Price',
    'r_364',
    'r_182',
    'days_Feb27',
    'PV_D1',
    'PV_D2',
    'PV_D',
    'Theo_Feb27'
]]

print("\n2027 Futures Output:")
print(df_2027.to_string())


2027 Futures Output:
         Date  Spot_Price     r_364     r_182  days_Feb27  PV_D1  PV_D2   PV_D  Theo_Feb27
0  2026-01-01      991.15  0.053911  0.054021         399  21.43   4.85  26.28     1023.44
1  2026-01-02     1001.60  0.053911  0.054021         398  21.43   4.85  26.29     1034.37
2  2026-01-05      977.50  0.053911  0.054021         395  21.44   4.85  26.30     1008.35
3  2026-01-06      962.20  0.053911  0.054021         394  21.45   4.85  26.30      991.98
4  2026-01-07      949.05  0.054309  0.054620         393  21.44   4.85  26.30      978.32
5  2026-01-08      946.70  0.054309  0.054620         392  21.45   4.85  26.30      975.68
6  2026-01-09      939.00  0.054309  0.054620         391  21.45   4.85  26.31      967.37
7  2026-01-12      936.95  0.054309  0.054620         388  21.46   4.86  26.32      964.75
8  2026-01-13      937.35  0.054309  0.054620         387  21.46   4.86  26.32      965.03
9  2026-01-14      925.45  0.054755  0.055195         386  21.46   4